In [ ]:
import hashlib
import numpy as np
import os
import pandas as pd
import re

from google.colab import drive
from openpyxl import load_workbook
from openpyxl.styles import Alignment

In [ ]:
# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# Set the path to the shortcut folder containing the result of each search (i.e., filter)
# Replace with the full path of your drive
search_history_folder = "/content/drive/MyDrive/.../FilterResults"

# Upload AllData file in your drive and then paste the link below
sheet_url = "<path-to-AllData-file>"
# Convert sheet to csv
csv_url = sheet_url.replace("/edit#gid=", "/gviz/tq?tqx=out:csv&gid=")

df_papers = pd.read_csv(csv_url)
df_papers.head()

,Author full names,Title,DOI,Link,Source,Abstract,Author Keywords,Index Keywords,To include
0,"Rehm, Johannes (57217101175); Reshodko, Irina ...",The virtual driving instructor: Multi-agent sy...,10.1002/aaai.12201,https://www.scopus.com/inward/record.uri?eid=2...,Scopus,"This work introduces the design, development, ...",NaN,Adversarial machine learning; Automobile drive...,False
1,"Li, Kaiyang (58835004200); Wang, Yaoying (5847...",Knowledge graph-based multimodal neural networ...,10.1002/eng2.12824,https://www.scopus.com/inward/record.uri?eid=2...,Scopus,The application of artificial intelligence in ...,defect detection; knowledge graph; multimodal ...,Computational efficiency; Defects; Graphic met...,False
2,"Lotz, Jeffrey C. (7102476007); Ropella, Glen (...",An exploration of knowledge-organizing technol...,10.1002/jsp2.1300,https://www.scopus.com/inward/record.uri?eid=2...,Scopus,Chronic low back pain (LBP) is influenced by a...,artificial intelligence; biopsychosocial model...,Article; artificial intelligence; biopsychosoc...,False
3,"Vassiliades, Alexandros (57216896352); Bassili...",A Knowledge Retrieval Framework for Household ...,10.1007/978-3-030-59833-4_3,https://www.scopus.com/inward/record.uri?eid=2...,Scopus,"In the field of domestic cognitive robotics, i...",Cognitive robotics; Knowledge retrieval framew...,Knowledge representation; Ontology; Cognitive ...,False
4,"Scrocca, Mario (57203516950); Comerio, Marco (...",Turning Transport Data to Comply with EU Stand...,10.1007/978-3-030-62466-8_26,https://www.scopus.com/inward/record.uri?eid=2...,Scopus,Complying with the EU Regulation on multimodal...,Multimodal transport knowledge graph; Semantic...,Data Sharing; Interoperability; Knowledge repr...,True


In [ ]:
# Set exclusion and inclusiong criteria and fields to look

# folder: Search_5579d07ec28f04090e1d3f1a06b3ec86

EXCLUSION_CRITERIA = [r"machine[- ]learning", r"neural[- ]networks?", r"gnns?"]
LOOK_TO_EXCLUDE = ["Title", "Author Keywords", "Index Keywords"]

INCLUSION_CRITERIA = [r"knowledge[- ]graphs?", r"ontolog.*"]
LOOK_TO_INCLUDE = ["Title", "Author Keywords", "Index Keywords", "Abstract"]

In [ ]:
# Set exclusion and inclusiong criteria and fields to look

# folder: Search_657f729b1df60f5b1af693a72ae6aafc
EXCLUSION_CRITERIA = [r"machine[- ]learning", r"neural[- ]networks?", r"gnns?", r"embeddings?"]
LOOK_TO_EXCLUDE = ["Title", "Author Keywords", "Index Keywords"]

INCLUSION_CRITERIA = [r"knowledge[- ]graphs?", r"ontolog.*",  r"multi-?modal|multimodality|modal.*"]
LOOK_TO_INCLUDE = ["Title", "Author Keywords", "Index Keywords", "Abstract"]

In [ ]:
# Set exclusion and inclusiong criteria and fields to look

# folder: Search_37ce23a73cdc6354f78b7384da6a6d18
EXCLUSION_CRITERIA = [r"machine[- ]learning", r"neural[- ]networks?", r"gnns?", r"embeddings?", r"deep[- ]learning"]
LOOK_TO_EXCLUDE = ["Title", "Author Keywords", "Index Keywords"]

INCLUSION_CRITERIA = [r"knowledge[- ]graphs?", r"ontolog.*",  r"multi-?modal|multimodality|modal.*"]
LOOK_TO_INCLUDE = ["Title", "Author Keywords", "Index Keywords", "Abstract"]

In [ ]:
# Set exclusion and inclusiong criteria and fields to look

# folder: Search_cee43462d54dccea7ca4afecaf95db52
EXCLUSION_CRITERIA = [
    r"machine[- ]learning", r"neural[- ]networks?", r"gnns?", r"embeddings?", r"deep[- ]learning", r"transformers?", r"lstm", r"convolution.*", r"knowledge fusion", r"bank", r"features?", r"dialog.*", r"question[- ]answering",
    r"information retrieval", r"adversarial", r"recommender", r"medical", r"molecular", r"drugs?", r"covid", r"legal", r"computer vision", r"object detector", r"contrastive learning", r"reinforcement learning", r"summarization", r"chatbots?",
    r"reviews?", r"surveys?", r"tools?"
    ]
LOOK_TO_EXCLUDE = ["Title", "Author Keywords", "Index Keywords"]

INCLUSION_CRITERIA = [r"knowledge[- ]graphs?", r"ontolog.*",  r"multi-?modal|multimodality|modal.*"]
LOOK_TO_INCLUDE = ["Title", "Author Keywords", "Index Keywords", "Abstract"]

In [ ]:
def add_wrapping_to_the_excel(excel_file_path):
    """
    Excel file with wrapped text in each cell.

    Parameters:
        excel_file_path (str): Path to the Excel file.
    """
    # Load the workbook and select the active sheet
    wb = load_workbook(excel_file_path)
    ws = wb.active

    # Apply text wrapping to all cells
    for row in ws.iter_rows():
        for cell in row:
            cell.alignment = Alignment(wrap_text=True, vertical='top')

    # Save the updated workbook with text wrapping
    wb.save(excel_file_path)


# Generate a unique search identifier (hash) based on criteria and fields
search_string = f"EXCLUDE:{EXCLUSION_CRITERIA}_INCLUDE:{INCLUSION_CRITERIA}_EXCLUDE_FIELDS:{LOOK_TO_EXCLUDE}_INCLUDE_FIELDS:{LOOK_TO_INCLUDE}"
search_hash = hashlib.md5(search_string.encode()).hexdigest()

# We check if this search has already be done
existing_folders = os.listdir(search_history_folder)
matching_folder = None

for folder in existing_folders:
    if search_hash in folder:
        matching_folder = folder
        break

if matching_folder:
    print(f"This search has already been performed. Check results in: {search_history_folder}/{matching_folder}")

    with open(f"{new_search_folder}/search_info.txt", "r") as f:
        search_info = f.read()
        print(search_info)

else:
    # If not

    # Create a new directory for this search
    new_search_folder = f"{search_history_folder}/Search_{search_hash}"
    os.makedirs(new_search_folder, exist_ok=True)

    # Convert values of columns to lowercase
    unique_fields = list(set(LOOK_TO_EXCLUDE + LOOK_TO_INCLUDE))
    df_papers[unique_fields] = df_papers[unique_fields].fillna("")
    df_papers_lower = df_papers.copy()[unique_fields]
    df_papers_lower[unique_fields] = df_papers_lower[unique_fields].apply(lambda x: x.astype(str).str.lower())

    # Remove rows with exclusion keywords
    mask_exclude = df_papers_lower[LOOK_TO_EXCLUDE].apply(lambda x: x.str.contains('|'.join(EXCLUSION_CRITERIA), na=False, regex=True)).any(axis=1)
    df_papers_excluded= df_papers_lower[mask_exclude]
    #df_papers_excluded.to_csv(f"{new_search_folder}/step1-excluded-by-exclusion_criteria.csv", index=False)

    path_to_papers_excluded = f"{new_search_folder}/step1-excluded-by-exclusion_criteria.xlsx"
    df_papers_excluded.to_excel(path_to_papers_excluded, index=False)
    add_wrapping_to_the_excel(path_to_papers_excluded)


    df_papers_not_excluded = df_papers_lower[~mask_exclude]

    # Keep only rows with inclusion keywords
    mask_include = df_papers_not_excluded[LOOK_TO_INCLUDE].apply(lambda x: x.str.contains('|'.join(INCLUSION_CRITERIA), na=False, regex=True)).any(axis=1)
    df_papers_not_included = df_papers_not_excluded[~mask_include]
    df_papers_survived = df_papers_not_excluded[mask_include]

    #df_papers_excluded.to_csv(f"{new_search_folder}/step2-excluded-by-inclusion_criteria.csv", index=False)
    path_to_papers_excluded_by_inclusion = f"{new_search_folder}/step2-excluded-by-inclusion_criteria.xlsx"
    df_papers_not_included.to_excel(path_to_papers_excluded_by_inclusion, index=False)
    add_wrapping_to_the_excel(path_to_papers_excluded_by_inclusion)

    #df_papers_survived.to_csv(f"{new_search_folder}/papers-survived.csv", index=False)
    path_to_papers_survived = f"{new_search_folder}/papers-survived.xlsx"
    df_papers_survived.to_excel(path_to_papers_survived, index=False)
    add_wrapping_to_the_excel(path_to_papers_survived)

    n_papers = len(df_papers)
    n_papers_excluded = len(df_papers_excluded)
    n_papers_not_excluded = len(df_papers_not_excluded)
    n_papers_not_included = len(df_papers_not_included)
    n_papers_survived = len(df_papers_survived)
    # Save search criteria info
    search_info = f"""

    Search Criteria Summary:
    =========================
    Exclusion Criteris: {EXCLUSION_CRITERIA}
    Columns Checked for Exclusion: {LOOK_TO_EXCLUDE}

    Inclusion Criteria: {INCLUSION_CRITERIA}
    Columns Checked for Inclusion: {LOOK_TO_INCLUDE}

    Total Papers Before Filtering: {n_papers}
    Total Papers After Exclusion Criteria: {n_papers_not_excluded} (-{n_papers_excluded})
    Total Papers After Inclusion Criteria: {n_papers_survived} (-{n_papers_not_included})

    Unique Search Hash: {search_hash}
    """

    print(f"Filtering completed! Results saved in: {new_search_folder}")
    with open(f"{new_search_folder}/search_info.txt", "w") as f:
        print(search_info)
        f.write(search_info)




This search has already been performed. Check results in: /content/drive/MyDrive/IDA/Projects/2025/MMPattern/ForJournal/FilterResults/Search_cee43462d54dccea7ca4afecaf95db52


    Search Criteria Summary:
    Exclusion Criteris: ['machine[- ]learning', 'neural[- ]networks?', 'gnns?', 'embeddings?', 'deep[- ]learning', 'transformers?', 'lstm', 'convolution.*', 'knowledge fusion', 'bank', 'features?', 'dialog.*', 'question[- ]answering', 'information retrieval', 'adversarial', 'recommender', 'medical', 'molecular', 'drugs?', 'covid', 'legal', 'computer vision', 'object detector', 'contrastive learning', 'reinforcement learning', 'summarization', 'chatbots?', 'reviews?', 'surveys?', 'tools?']
    Columns Checked for Exclusion: ['Title', 'Author Keywords', 'Index Keywords']

    Inclusion Criteria: ['knowledge[- ]graphs?', 'ontolog.*', 'multi-?modal|multimodality|modal.*']
    Columns Checked for Inclusion: ['Title', 'Author Keywords', 'Index Keywords', 'Abstract']

    Total Papers Before